# Powerful Agents — LangGraph vs Strands vs CrewAI

**Classroom demo notebook.** Build the *same* capable agent three ways and compare how each framework is assembled.

| You will build | Shape |
|---|---|
| **LangGraph** power agent | ReAct / tool-calling loop (`create_agent`) |
| **Strands** power agent | Single `Agent` + tool list |
| **CrewAI** power crew | Researcher → Analyst → Writer |

### Capability stack (all three)
Tools · FAQ + RAG · calculator · datetime · web search · (optional) MCP Gateway · (optional) Memory · Browser/Harness explained

> Run top → bottom. OpenAI key required for live demos. AgentCore cells are **optional** and skip cleanly if env vars are missing.


## 0) Install & environment


In [1]:
# 👇 What this cell does: install notebook deps (Colab/local). Prefer `uv sync` from the lab folder when local.
%pip install -q python-dotenv openai httpx ddgs tavily-python \
  langchain langchain-core langchain-openai langgraph \
  "strands-agents>=1.0.0" strands-agents-tools \
  "crewai>=0.80.0" boto3



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import csv
import os
import re
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

# 👇 What this does: load .env from this lab folder (OPENAI_API_KEY, optional AgentCore vars)
load_dotenv(Path(".env"))
if not os.getenv("OPENAI_API_KEY"):
    load_dotenv(Path("../01.langraph-agentcore-bedrock/.env"))  # optional fallback while teaching

DATA = Path("data")
FAQ_PATH = DATA / "lauki_qna.csv"
RAG_PATH = DATA / "agentcore_knowledge.md"
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

assert FAQ_PATH.exists(), f"Missing {FAQ_PATH}"
assert RAG_PATH.exists(), f"Missing {RAG_PATH}"
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

print("✅ OpenAI key loaded")
print("✅ FAQ:", FAQ_PATH, "| RAG:", RAG_PATH)
print("Optional MEMORY_ID:", bool(os.getenv("MEMORY_ID")))
print("Optional GATEWAY_URL:", bool(os.getenv("GATEWAY_URL")))
print("Optional GATEWAY_TOKEN:", bool(os.getenv("GATEWAY_TOKEN")))


✅ OpenAI key loaded
✅ FAQ: data/lauki_qna.csv | RAG: data/agentcore_knowledge.md
Optional MEMORY_ID: False
Optional GATEWAY_URL: False
Optional GATEWAY_TOKEN: False


## 1) Shared knowledge layer (framework-agnostic)

Analogy: the **library** is the same; LangGraph / Strands / CrewAI are three different **librarians**.


In [3]:
def _tokenize(text: str) -> set[str]:
    # 👇 Simple token set for keyword overlap retrieval (no embedding quota)
    return {t for t in re.findall(r"[a-z0-9]+", text.lower()) if len(t) > 2}


def load_faq(path: Path = FAQ_PATH) -> list[dict[str, str]]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            rows.append({"question": row["question"].strip(), "answer": row["answer"].strip()})
    return rows


def load_rag_chunks(path: Path = RAG_PATH, max_chars: int = 900) -> list[str]:
    text = path.read_text(encoding="utf-8")
    # Split on headings / blank lines into rough chunks
    parts = re.split(r"\n(?=#)|\n\n+", text)
    chunks = [p.strip() for p in parts if len(p.strip()) > 40]
    return [c[:max_chars] for c in chunks]


FAQ_ROWS = load_faq()
RAG_CHUNKS = load_rag_chunks()
print(f"FAQ rows: {len(FAQ_ROWS)} | RAG chunks: {len(RAG_CHUNKS)}")


def search_knowledge(query: str, k: int = 4) -> str:
    """Unified RAG+FAQ retrieval used by all three frameworks."""
    q = _tokenize(query)
    scored: list[tuple[int, str]] = []
    for row in FAQ_ROWS:
        blob = f"FAQ Q: {row['question']}\nA: {row['answer']}"
        score = len(q & _tokenize(blob))
        if score:
            scored.append((score, blob))
    for chunk in RAG_CHUNKS:
        score = len(q & _tokenize(chunk))
        if score:
            scored.append((score, f"DOC:\n{chunk}"))
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [t for _, t in scored[:k]]
    if not top:
        return "No relevant knowledge found."
    return "Knowledge results:\n\n" + "\n\n---\n\n".join(top)


# Smoke test
print(search_knowledge("Does Lauki support eSIM?")[:500])


FAQ rows: 75 | RAG chunks: 9
Knowledge results:

FAQ Q: Does Lauki Phones support eSIM?
A: Supported on devices that follow GSMA eSIM standards. Provisioning occurs through the Lauki Phones portal, which issues a single-use QR profile tied to the account.

---

FAQ Q: Does Lauki Phones support VoLTE?
A: VoLTE is enabled for devices certified on Lauki Phones’ IMS network. High-definition calling activates automatically when LTE coverage is available.

---

FAQ Q: Does Lauki Phones support WiFi Calling?
A: Supported for most 


## 2) Mental model — how each framework is *built*

| Concern | LangGraph | Strands | CrewAI |
|---|---|---|---|
| Unit of work | Graph node / agent loop | One `Agent` | `Agent` + `Task` + `Crew` |
| Tools | `@tool` (LangChain) on one agent | `@tool` (Strands) on one agent | `@tool` (CrewAI); often on Researcher only |
| Control flow | Edges, retries, supervisors | Model decides tool calls | Sequential / hierarchical process |
| Multi-step | Explicit graph or ReAct loop | Single-agent tool loop | Multiple specialists + `context=[]` |
| AgentCore Memory | Checkpointer / Store | `AgentCoreMemorySessionManager` | Manual `MemoryClient` events |
| AgentCore MCP | LangChain `StructuredTool` wrappers | Keep `MCPClient` alive + tools | Re-wrap as CrewAI tools |

**Rule of thumb for the demo:** same *capabilities*, different *orchestration*.


---
## 3) LangGraph — power agent (`create_agent` + tools + RAG)

**Analogy:** One skilled operator with a toolbelt and a playbook (system prompt). LangGraph runs the Think → Act → Observe loop.


In [4]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# 👇 LangChain @tool — schema comes from type hints + docstring
@tool
def lg_search_knowledge(query: str) -> str:
    """Search Lauki FAQ + AgentCore knowledge docs for grounded answers."""
    return search_knowledge(query, k=4)


@tool
def lg_calculator(expression: str) -> str:
    """Evaluate a simple math expression like '15*8+3'."""
    allowed = set("0123456789+-*/().% ")
    if not expression or any(c not in allowed for c in expression):
        return "Only simple arithmetic is allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))  # noqa: S307 — demo sandbox
    except Exception as exc:  # noqa: BLE001
        return f"Math error: {exc}"


@tool
def lg_now() -> str:
    """Return current UTC datetime."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


@tool
def lg_web_search(query: str) -> str:
    """Search the public web when local knowledge is not enough."""
    tavily = os.getenv("TAVILY_API_KEY", "").strip()
    if tavily:
        from tavily import TavilyClient
        hits = TavilyClient(api_key=tavily).search(query, max_results=3).get("results", [])
        return "\n".join(f"- {h.get('title')}: {h.get('content', '')[:220]}" for h in hits) or "No hits"
    try:
        from ddgs import DDGS
        hits = list(DDGS().text(query, max_results=3))
        return "\n".join(f"- {h.get('title')}: {h.get('body', '')[:220]}" for h in hits) or "No hits"
    except Exception as exc:  # noqa: BLE001
        return f"web_search unavailable: {exc}"


LG_TOOLS = [lg_search_knowledge, lg_calculator, lg_now, lg_web_search]
lg_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2)

LG_SYSTEM = (
    "You are a powerful research+support agent.\n"
    "Prefer lg_search_knowledge for Lauki / AgentCore facts.\n"
    "Use lg_calculator for math, lg_now for time, lg_web_search only if needed.\n"
    "Be concise and cite which tool you used."
)

# 👇 create_agent wires model + tools into a runnable ReAct-style agent
langgraph_agent = create_agent(model=lg_llm, tools=LG_TOOLS, system_prompt=LG_SYSTEM)
print("✅ LangGraph power agent ready | tools:", [t.name for t in LG_TOOLS])


✅ LangGraph power agent ready | tools: ['lg_search_knowledge', 'lg_calculator', 'lg_now', 'lg_web_search']


In [5]:
def run_langgraph(prompt: str) -> str:
    # 👇 Standard LangChain messages payload for create_agent
    result = langgraph_agent.invoke({"messages": [("human", prompt)]})
    messages = result.get("messages") or []
    last = messages[-1]
    content = getattr(last, "content", last)
    if isinstance(content, list):
        # Some models return content blocks
        content = " ".join(
            (c.get("text") if isinstance(c, dict) else str(c)) for c in content
        )
    return str(content)


demo_q = "Does Lauki support eSIM? Also what is 12*9+4?"
print("Q:", demo_q)
print("A:", run_langgraph(demo_q))


Q: Does Lauki support eSIM? Also what is 12*9+4?
A: Yes, Lauki supports eSIM on devices that follow GSMA eSIM standards. Provisioning occurs through the Lauki Phones portal, which issues a single-use QR profile tied to the account.

The result of the calculation \(12 \times 9 + 4\) is 112.


---
## 4) Strands — power agent (single `Agent` + tools)

**Analogy:** One focused specialist. You hand them tools + a system prompt; Strands handles the tool loop.


In [6]:
from strands import Agent, tool
from strands.models.openai import OpenAIModel

# 👇 Strands @tool — same idea, different package
@tool
def st_search_knowledge(query: str) -> str:
    """Search Lauki FAQ + AgentCore knowledge docs for grounded answers."""
    return search_knowledge(query, k=4)


@tool
def st_calculator(expression: str) -> str:
    """Evaluate a simple math expression like '15*8+3'."""
    allowed = set("0123456789+-*/().% ")
    if not expression or any(c not in allowed for c in expression):
        return "Only simple arithmetic is allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))  # noqa: S307
    except Exception as exc:  # noqa: BLE001
        return f"Math error: {exc}"


@tool
def st_now() -> str:
    """Return current UTC datetime."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


@tool
def st_web_search(query: str) -> str:
    """Search the public web when local knowledge is not enough."""
    try:
        from ddgs import DDGS
        hits = list(DDGS().text(query, max_results=3))
        return "\n".join(f"- {h.get('title')}: {h.get('body', '')[:220]}" for h in hits) or "No hits"
    except Exception as exc:  # noqa: BLE001
        return f"web_search unavailable: {exc}"


ST_TOOLS = [st_search_knowledge, st_calculator, st_now, st_web_search]
st_model = OpenAIModel(
    client_args={"api_key": os.environ["OPENAI_API_KEY"]},
    model_id=OPENAI_MODEL,
)

ST_SYSTEM = (
    "You are Lauki + AgentCore support/research copilot.\n"
    "Prefer st_search_knowledge first. Use calculator/time/web when needed.\n"
    "Be concise."
)

# 👇 Strands Agent = model + tools + system prompt (no explicit graph)
strands_agent = Agent(model=st_model, tools=ST_TOOLS, system_prompt=ST_SYSTEM)
print("✅ Strands power agent ready")


✅ Strands power agent ready


In [7]:
def _unwrap_text(val) -> str:
    """Flatten Strands/OpenAI-style content blocks into plain text."""
    if val is None:
        return ""
    if isinstance(val, str):
        return val
    if isinstance(val, list):
        parts = []
        for item in val:
            if isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(_unwrap_text(item))
        return "\n".join(p for p in parts if p)
    if isinstance(val, dict):
        if "text" in val:
            return str(val["text"])
        if "content" in val:
            return _unwrap_text(val["content"])
        return str(val)
    return str(val)


def strands_text(result) -> str:
    # Normalize Strands AgentResult / message blocks for clean demo output
    if result is None:
        return ""
    if isinstance(result, str):
        return result
    for attr in ("message", "content", "text", "output"):
        if hasattr(result, attr):
            text = _unwrap_text(getattr(result, attr))
            if text.strip():
                return text
    return _unwrap_text(result)


def run_strands(prompt: str) -> str:
    return strands_text(strands_agent(prompt))


print("Q:", demo_q)
print("A:", run_strands(demo_q))


Q: Does Lauki support eSIM? Also what is 12*9+4?

Tool #1: st_search_knowledge

Tool #2: st_calculator
Yes, Lauki supports eSIM on devices that follow GSMA eSIM standards. Provisioning occurs through the Lauki Phones portal, which issues a single-use QR profile tied to the account.

The result of the calculation \(12 \times 9 + 4\) is \(112\).A: Yes, Lauki supports eSIM on devices that follow GSMA eSIM standards. Provisioning occurs through the Lauki Phones portal, which issues a single-use QR profile tied to the account.

The result of the calculation \(12 \times 9 + 4\) is \(112\).


---
## 5) CrewAI — power crew (multi-agent pipeline)

**Analogy:** A small consulting firm — Researcher gathers, Analyst interprets, Writer delivers the brief.
This is the biggest structural difference vs LangGraph/Strands single-agent loops.


In [8]:
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")

from crewai import Agent as CrewAgent, Crew, LLM, Process, Task
from crewai.tools import BaseTool
from pydantic import BaseModel, Field

# Explicit args_schema — prevents CrewAI from sending description/metadata as tool args


class KnowledgeQuery(BaseModel):
    query: str = Field(..., description="Search query text for FAQ / AgentCore docs")


class CalcInput(BaseModel):
    expression: str = Field(..., description="Simple arithmetic expression, e.g. 12*9+4")


class WebQuery(BaseModel):
    query: str = Field(..., description="Web search query")


class SearchKnowledgeTool(BaseTool):
    name: str = "search_knowledge"
    description: str = (
        "Search Lauki FAQ + AgentCore knowledge docs. "
        'Pass ONLY {"query": "your search text"}.'
    )
    args_schema: type[BaseModel] = KnowledgeQuery

    def _run(self, query: str) -> str:
        return search_knowledge(query, k=4)


class CalculatorTool(BaseTool):
    name: str = "calculator"
    description: str = 'Evaluate simple math. Pass ONLY {"expression": "12*9+4"}.'
    args_schema: type[BaseModel] = CalcInput

    def _run(self, expression: str) -> str:
        allowed = set("0123456789+-*/().% ")
        if not expression or any(c not in allowed for c in expression):
            return "Only simple arithmetic is allowed."
        try:
            return str(eval(expression, {"__builtins__": {}}, {}))  # noqa: S307
        except Exception as exc:  # noqa: BLE001
            return f"Math error: {exc}"


class WebSearchTool(BaseTool):
    name: str = "web_search"
    description: str = 'Search the public web. Pass ONLY {"query": "your search text"}.'
    args_schema: type[BaseModel] = WebQuery

    def _run(self, query: str) -> str:
        try:
            from ddgs import DDGS

            hits = list(DDGS().text(query, max_results=3))
            return "\n".join(
                f"- {h.get('title')}: {h.get('body', '')[:220]}" for h in hits
            ) or "No hits"
        except Exception as exc:  # noqa: BLE001
            return f"web_search unavailable: {exc}"


CREW_TOOLS = [SearchKnowledgeTool(), CalculatorTool(), WebSearchTool()]
crew_llm = LLM(model=f"openai/{OPENAI_MODEL}", api_key=os.environ["OPENAI_API_KEY"])


def build_power_crew(topic: str) -> Crew:
    researcher = CrewAgent(
        role="Knowledge Researcher",
        goal="Gather grounded facts using search_knowledge and calculator tools",
        backstory="You retrieve accurate product and platform facts quickly.",
        llm=crew_llm,
        tools=CREW_TOOLS,
        verbose=False,  # quiet — Jupyter widget renderer breaks on CrewAI rich UI
        allow_delegation=False,
        max_iter=4,
    )
    analyst = CrewAgent(
        role="Insights Analyst",
        goal="Turn research into clear strengths, gaps, and implications",
        backstory="You compare options and highlight what matters for a demo audience.",
        llm=crew_llm,
        tools=[],
        verbose=False,
        allow_delegation=False,
        max_iter=3,
    )
    writer = CrewAgent(
        role="Demo Brief Writer",
        goal="Write a crisp demo-ready answer",
        backstory="You write short, clear answers with sections and bullets.",
        llm=crew_llm,
        tools=[],
        verbose=False,
        allow_delegation=False,
        max_iter=3,
    )

    t1 = Task(
        description=(
            f"Research this user request: {topic}\n\n"
            "Tool rules:\n"
            "- Call search_knowledge with argument query=<short search string>\n"
            "- Call calculator with argument expression=<math> when there is math\n"
            "- Use web_search only if local knowledge is empty\n"
            "Do NOT pass description/metadata objects as tool arguments."
        ),
        expected_output="Bullet research notes with tools used.",
        agent=researcher,
    )
    t2 = Task(
        description="Analyze the research. List key facts, risks, and 3 takeaways.",
        expected_output="Structured analysis bullets.",
        agent=analyst,
        context=[t1],
    )
    t3 = Task(
        description=(
            "Write the final answer for the user. "
            "Sections: Answer, Evidence, Next step. Keep it under 250 words."
        ),
        expected_output="Final markdown answer.",
        agent=writer,
        context=[t2],
    )
    return Crew(
        agents=[researcher, analyst, writer],
        tasks=[t1, t2, t3],
        process=Process.sequential,
        verbose=False,
    )


print("✅ CrewAI power-crew builder ready (BaseTool schemas + quiet mode)")


✅ CrewAI power-crew builder ready (BaseTool schemas + quiet mode)


In [9]:
def run_crewai(prompt: str) -> str:
    crew = build_power_crew(prompt)
    # kickoff runs Researcher → Analyst → Writer in order
    out = crew.kickoff(inputs={"topic": prompt})
    return str(getattr(out, "raw", None) or out)


crew_q = "Does Lauki support eSIM? Summarize AgentCore Memory in 2 bullets. What is 12*9+4?"
print("Q:", crew_q)
print("A:", run_crewai(crew_q))


Q: Does Lauki support eSIM? Summarize AgentCore Memory in 2 bullets. What is 12*9+4?
A: ### Answer
Lauki enhances user experience and device connectivity through its support for GSMA eSIM standards. By enabling easy provisioning via a single-use QR code through the Lauki Phones portal, users can efficiently activate their eSIMs. The system employs an advanced memory management structure within AgentCore, which organizes data with unique identifiers (`actor_id` + `thread_id`), ensuring effective handling of multi-turn chats.

### Evidence
- Lauki's eSIM implementation aligns with industry standards, facilitating seamless connectivity.
- The provisioning mechanism via QR codes allows quick activation while also presenting potential security risks if not secured properly.
- The memory management tools, such as `AgentCoreMemorySaver` and `MemoryMiddleware`, are critical for context maintenance in interactions.
- A miscalculation example (12*9+4=112) highlights the importance of accuracy in

---
## 6) Side-by-side bake-off (same prompt)

Run the **same** user question through all three stacks. Great live-class moment.


In [10]:
BAKEOFF_PROMPT = (
    "A customer asks: how do I activate a new SIM, and does Lauki support eSIM? "
    "Also compute 25*4-10. Keep the answer short."
)

results = {}
print("⏳ LangGraph...")
results["LangGraph"] = run_langgraph(BAKEOFF_PROMPT)
print("⏳ Strands...")
results["Strands"] = run_strands(BAKEOFF_PROMPT)
print("⏳ CrewAI (slower — 3 agents)...")
results["CrewAI"] = run_crewai(BAKEOFF_PROMPT)

for name, answer in results.items():
    print("\n" + "=" * 72)
    print(name)
    print("=" * 72)
    print(answer[:1200])


⏳ LangGraph...
⏳ Strands...

Tool #3: st_search_knowledge

Tool #4: st_search_knowledge

Tool #5: st_calculator
To activate a new SIM with Lauki, insert the SIM, complete KYC with a government-issued ID, verify phone number ownership, and restart your device. Network registration finishes once identity and address checks are cleared.

Yes, Lauki supports eSIM on devices following GSMA eSIM standards.

The result of \(25 \times 4 - 10\) is \(90\).⏳ CrewAI (slower — 3 agents)...

LangGraph
To activate a new SIM, insert it, complete KYC with a government-issued ID, verify phone ownership, and restart your device. Network registration will finish once identity checks clear.

Yes, Lauki supports eSIM on devices that follow GSMA standards, with provisioning done through the Lauki Phones portal.

The result of 25*4-10 is 90.

Strands
To activate a new SIM with Lauki, insert the SIM, complete KYC with a government-issued ID, verify phone number ownership, and restart your device. Network regis

---
## 7) Optional — AgentCore deep features (MCP Gateway + Memory)

These cells mirror what labs **01 / 02 / 03** deploy on Runtime.  
They **skip** if `GATEWAY_*` / `MEMORY_ID` are missing so the notebook still demos offline.


In [11]:
def try_list_gateway_tools() -> None:
    """Call AgentCore Gateway MCP tools/list if GATEWAY_URL + GATEWAY_TOKEN are set."""
    url = (os.getenv("GATEWAY_URL") or "").strip()
    token = (os.getenv("GATEWAY_TOKEN") or "").strip()
    if not url or not token:
        print("⏭️  Skip MCP Gateway — set GATEWAY_URL and GATEWAY_TOKEN to demo.")
        print("   Create them with: 01/02/03 scripts/create_mcp_gateway.py + get_gateway_token.py")
        return
    import httpx
    r = httpx.post(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream",
        },
        json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}},
        timeout=60,
    )
    print("Gateway status:", r.status_code)
    print(r.text[:900])


try_list_gateway_tools()


⏭️  Skip MCP Gateway — set GATEWAY_URL and GATEWAY_TOKEN to demo.
   Create them with: 01/02/03 scripts/create_mcp_gateway.py + get_gateway_token.py


In [12]:
def explain_memory_wiring():
    """How each framework attaches AgentCore Memory (reference — no cloud write)."""
    print(
        """
MEMORY WIRING CHEAT SHEET
=========================
LangGraph (lab 01):
  AgentCoreMemorySaver + AgentCoreMemoryStore + MemoryMiddleware
  create_agent(..., checkpointer=..., store=..., middleware=[MemoryMiddleware()])

Strands (lab 02):
  AgentCoreMemoryConfig(memory_id=MEMORY_ID, ...)
  AgentCoreMemorySessionManager(...)
  Agent(..., session_manager=session_manager)

CrewAI (lab 03):
  MemoryClient().list_events / create_event
  Prefetch load_memory_context(actor_id, session_id) into task text
  After kickoff: save_memory_turn(...)

Common invoke payload fields: actor_id + thread_id (session)
"""
    )
    mid = (os.getenv("MEMORY_ID") or "").strip()
    if len(mid) >= 12:
        print(f"✅ MEMORY_ID present ({mid[:12]}...) — use Runtime demos in labs 01–03 for live recall.")
    else:
        print("⏭️  No MEMORY_ID — Memory live demo stays in labs 01–03 after Step 1.")


explain_memory_wiring()



MEMORY WIRING CHEAT SHEET
LangGraph (lab 01):
  AgentCoreMemorySaver + AgentCoreMemoryStore + MemoryMiddleware
  create_agent(..., checkpointer=..., store=..., middleware=[MemoryMiddleware()])

Strands (lab 02):
  AgentCoreMemoryConfig(memory_id=MEMORY_ID, ...)
  AgentCoreMemorySessionManager(...)
  Agent(..., session_manager=session_manager)

CrewAI (lab 03):
  MemoryClient().list_events / create_event
  Prefetch load_memory_context(actor_id, session_id) into task text
  After kickoff: save_memory_turn(...)

Common invoke payload fields: actor_id + thread_id (session)

⏭️  No MEMORY_ID — Memory live demo stays in labs 01–03 after Step 1.


---
## 8) Browser + deep Harness (LangGraph / AgentCore specialty)

| Feature | Where it lives | What students should remember |
|---|---|---|
| **Browser tool** (`browse_url`) | Lab **01** research StateGraph | Cloud Chromium via AgentCore Browser + Playwright CDP |
| **Code Interpreter** | Lab **01** Demo 5 | Your agent calls `run_python_code` tool |
| **Managed Harness** | Lab **01** Demo 6 | AWS owns the loop — `InvokeHarness`, **no** `langraph_*.py` deploy |

Strands / CrewAI labs intentionally **reuse** Memory/Gateway/Identity but leave Harness in **01**.

### Demo talking point
> LangGraph is where we show *deep* AgentCore (Browser + Harness).  
> Strands shows the cleanest single-agent Support Copilot.  
> CrewAI shows multi-agent division of labor for briefs.


In [13]:
# 👇 Conceptual Harness/Browser sketch (does not call AWS unless you extend it)
print(
    """
DEEP AGENT STACK (LangGraph + AgentCore)
---------------------------------------
1. Tools: FAQ/RAG + calculator + web_search
2. MCP Gateway tools: get_weather / get_time (JWT or Identity)
3. Memory: actor_id + thread_id across turns
4. Browser: browse_url → BrowserClient.start → Playwright over CDP
5. Code Interpreter: sandboxed Python for exact computation
6. Managed Harness: AWS loop for "just give me an agent" demos

Classroom path:
  Notebook (this file)  →  compare frameworks locally
  Lab 01 Runtime demos →  put the same ideas on AgentCore cloud
  Lab 02 / 03          →  same AgentCore arc, different frameworks
"""
)



DEEP AGENT STACK (LangGraph + AgentCore)
---------------------------------------
1. Tools: FAQ/RAG + calculator + web_search
2. MCP Gateway tools: get_weather / get_time (JWT or Identity)
3. Memory: actor_id + thread_id across turns
4. Browser: browse_url → BrowserClient.start → Playwright over CDP
5. Code Interpreter: sandboxed Python for exact computation
6. Managed Harness: AWS loop for "just give me an agent" demos

Classroom path:
  Notebook (this file)  →  compare frameworks locally
  Lab 01 Runtime demos →  put the same ideas on AgentCore cloud
  Lab 02 / 03          →  same AgentCore arc, different frameworks



---
## 9) Decision guide — which framework for the demo?

| If you want to show… | Pick |
|---|---|
| Explicit graphs, retries, supervisors, Browser/Harness | **LangGraph** (lab 01) |
| Fast single specialist + tools on AgentCore | **Strands** (lab 02) |
| Multi-role team producing a brief | **CrewAI** (lab 03) |
| Side-by-side architecture lesson | **This notebook** |

### Challenge (optional)
1. Add an MCP Gateway tool into **one** framework only and re-run the bake-off.  
2. Add a fourth “router” that sends support questions to Strands and brief topics to CrewAI.  
3. Deploy the winner to AgentCore Runtime using the matching lab README (Steps 0→3 first).
